In [ ]:
from __future__ import annotations

from getpass import getpass
from pathlib import Path

import pandas as pd
from sqlalchemy import URL, create_engine, text
from sqlalchemy.engine import Engine


# ============================================================
# 1. CONFIGURATION
# ============================================================

DB_PASSWORD = getpass("Enter PostgreSQL password: ")

DATABASE_URL = URL.create(
    drivername="postgresql+psycopg",
    username="postgres",
    password=DB_PASSWORD,
    host="localhost",
    port=5433,
    database="olist",
)

engine: Engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,
)

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

ML_TABLE_PATH = ARTIFACT_DIR / "01_ml_table.csv"
AUDIT_PATH = ARTIFACT_DIR / "01_table_audit.csv"


# ============================================================
# 2. TABLE DEFINITIONS
# ============================================================

TABLE_INFO = {
    "olist_orders_dataset": {
        "key": "order_id",
        "row_meaning": "One row per order",
    },
    "olist_customers_dataset": {
        "key": "customer_id",
        "row_meaning": "One row per order-specific customer record",
    },
    "olist_order_items_dataset": {
        "key": "(order_id, order_item_id)",
        "row_meaning": "One row per item inside an order",
    },
    "olist_order_payments_dataset": {
        "key": "(order_id, payment_sequential)",
        "row_meaning": "One row per payment record for an order",
    },
    "olist_order_reviews_dataset": {
    "key": "(review_id, order_id)",
    "row_meaning": "One row per review-order pair",
},
    "olist_products_dataset": {
        "key": "product_id",
        "row_meaning": "One row per product",
    },
    "olist_sellers_dataset": {
        "key": "seller_id",
        "row_meaning": "One row per seller",
    },
    "olist_geolocation_dataset": {
        "key": None,
        "row_meaning": "One geolocation observation for a ZIP-code prefix",
    },
    "product_category_name_translation": {
        "key": "product_category_name",
        "row_meaning": "One Portuguese-to-English category mapping",
    },
}


# ============================================================
# 3. DATABASE CONNECTION TEST
# ============================================================

with engine.connect() as connection:
    result = connection.execute(text("SELECT 1;")).scalar_one()

assert result == 1

print("PostgreSQL connection: PASS")


# ============================================================
# 4. AUDIT ALL TABLES
# ============================================================

audit_rows: list[dict[str, object]] = []

for table_name, info in TABLE_INFO.items():

    with engine.connect() as connection:

        row_count = connection.execute(
            text(f"SELECT COUNT(*) FROM public.{table_name};")
        ).scalar_one()

        key_expression = info["key"]

        if key_expression is None:
            duplicate_rows = None

        else:
            unique_count = connection.execute(
                text(
                    f"""
                    SELECT COUNT(DISTINCT {key_expression})
                    FROM public.{table_name};
                    """
                )
            ).scalar_one()

            duplicate_rows = row_count - unique_count

    audit_rows.append(
        {
            "table_name": table_name,
            "row_count": row_count,
            "key": key_expression,
            "duplicate_key_rows": duplicate_rows,
            "row_meaning": info["row_meaning"],
        }
    )


audit_df = pd.DataFrame(audit_rows)

print("\nTABLE AUDIT")
print("=" * 100)

display(audit_df)

audit_df.to_csv(
    AUDIT_PATH,
    index=False,
)


# ============================================================
# 5. LOOK AT EACH TABLE
# ============================================================

for table_name in TABLE_INFO:

    print("\n")
    print("=" * 100)
    print(table_name)
    print("=" * 100)

    sample_df = pd.read_sql(
        f"""
        SELECT *
        FROM public.{table_name}
        LIMIT 3;
        """,
        engine,
    )

    display(sample_df)


# ============================================================
# 6. BUILD ONE-ROW-PER-ORDER ML TABLE
# ============================================================
#
# IMPORTANT:
#
# order_items has many rows per order.
# payments has many rows per order.
# sellers/products may also appear many times through items.
# geolocation has many rows per ZIP prefix.
#
# Therefore we aggregate FIRST and join AFTERWARD.
#
# Reviews are inspected above but intentionally NOT included
# in the ML table because review information is created after
# the order/delivery process and would create future leakage.
# ============================================================

ml_query = """
WITH geo AS (
    SELECT
        geolocation_zip_code_prefix,
        AVG(geolocation_lat) AS avg_lat,
        AVG(geolocation_lng) AS avg_lng
    FROM public.olist_geolocation_dataset
    GROUP BY geolocation_zip_code_prefix
),

product_dim AS (
    SELECT
        p.product_id,
        COALESCE(
            t.product_category_name_english,
            p.product_category_name
        ) AS product_category,
        p.product_weight_g,
        p.product_length_cm,
        p.product_height_cm,
        p.product_width_cm
    FROM public.olist_products_dataset AS p

    LEFT JOIN public.product_category_name_translation AS t
        ON p.product_category_name = t.product_category_name
),

item_agg AS (
    SELECT
        oi.order_id,

        COUNT(*) AS item_count,

        COUNT(DISTINCT oi.product_id)
            AS unique_product_count,

        COUNT(DISTINCT oi.seller_id)
            AS unique_seller_count,

        SUM(oi.price)
            AS total_item_price,

        AVG(oi.price)
            AS avg_item_price,

        MAX(oi.price)
            AS max_item_price,

        SUM(oi.freight_value)
            AS total_freight_value,

        AVG(pd.product_weight_g)
            AS avg_product_weight_g,

        AVG(pd.product_length_cm)
            AS avg_product_length_cm,

        AVG(pd.product_height_cm)
            AS avg_product_height_cm,

        AVG(pd.product_width_cm)
            AS avg_product_width_cm,

        STRING_AGG(
            DISTINCT pd.product_category,
            ' | '
        ) AS product_categories,

        STRING_AGG(
            DISTINCT s.seller_state,
            ' | '
        ) AS seller_states,

        STRING_AGG(
            DISTINCT s.seller_city,
            ' | '
        ) AS seller_cities,

        AVG(sg.avg_lat)
            AS avg_seller_lat,

        AVG(sg.avg_lng)
            AS avg_seller_lng

    FROM public.olist_order_items_dataset AS oi

    LEFT JOIN product_dim AS pd
        ON oi.product_id = pd.product_id

    LEFT JOIN public.olist_sellers_dataset AS s
        ON oi.seller_id = s.seller_id

    LEFT JOIN geo AS sg
        ON s.seller_zip_code_prefix =
           sg.geolocation_zip_code_prefix

    GROUP BY oi.order_id
),

payment_agg AS (
    SELECT
        order_id,

        COUNT(*) AS payment_record_count,

        COUNT(DISTINCT payment_type)
            AS payment_type_count,

        STRING_AGG(
            DISTINCT payment_type,
            ' | '
        ) AS payment_types,

        MAX(payment_installments)
            AS max_payment_installments,

        SUM(payment_value)
            AS total_payment_value

    FROM public.olist_order_payments_dataset

    GROUP BY order_id
)

SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,

    o.order_status,

    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    c.customer_zip_code_prefix,
    c.customer_city,
    c.customer_state,

    cg.avg_lat AS customer_lat,
    cg.avg_lng AS customer_lng,

    ia.item_count,
    ia.unique_product_count,
    ia.unique_seller_count,

    ia.total_item_price,
    ia.avg_item_price,
    ia.max_item_price,
    ia.total_freight_value,

    ia.avg_product_weight_g,
    ia.avg_product_length_cm,
    ia.avg_product_height_cm,
    ia.avg_product_width_cm,

    ia.product_categories,

    ia.seller_states,
    ia.seller_cities,

    ia.avg_seller_lat,
    ia.avg_seller_lng,

    pa.payment_record_count,
    pa.payment_type_count,
    pa.payment_types,
    pa.max_payment_installments,
    pa.total_payment_value

FROM public.olist_orders_dataset AS o

LEFT JOIN public.olist_customers_dataset AS c
    ON o.customer_id = c.customer_id

LEFT JOIN geo AS cg
    ON c.customer_zip_code_prefix =
       cg.geolocation_zip_code_prefix

LEFT JOIN item_agg AS ia
    ON o.order_id = ia.order_id

LEFT JOIN payment_agg AS pa
    ON o.order_id = pa.order_id

ORDER BY o.order_purchase_timestamp;
"""


ml_df = pd.read_sql(
    ml_query,
    engine,
)


# ============================================================
# 7. VALIDATE FINAL GRAIN
# ============================================================

order_count = pd.read_sql(
    """
    SELECT COUNT(*) AS count
    FROM public.olist_orders_dataset;
    """,
    engine,
)["count"].iloc[0]

assert len(ml_df) == order_count, (
    f"Row-count mismatch: ML table has {len(ml_df):,} rows "
    f"but orders table has {order_count:,}."
)

duplicate_order_ids = ml_df["order_id"].duplicated().sum()

assert duplicate_order_ids == 0, (
    f"Found {duplicate_order_ids} duplicate order_id values."
)


print("\nFINAL ML TABLE VALIDATION")
print("=" * 100)

print(f"Orders table rows: {order_count:,}")
print(f"ML table rows:     {len(ml_df):,}")
print(f"Unique order IDs:  {ml_df['order_id'].nunique():,}")
print(f"Duplicate orders:  {duplicate_order_ids}")

print("\nFinal shape:")
print(ml_df.shape)

print("\nColumns:")
print(list(ml_df.columns))


# ============================================================
# 8. FINAL SAMPLE
# ============================================================

display(
    ml_df.head(10)
)


# ============================================================
# 9. NULL SUMMARY
# ============================================================

null_summary = (
    ml_df.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

null_summary["missing_percentage"] = (
    null_summary["missing_count"]
    / len(ml_df)
    * 100
)

print("\nMISSING VALUE SUMMARY")
print("=" * 100)

display(
    null_summary.head(20)
)


# ============================================================
# 10. SAVE ARTIFACT
# ============================================================

ml_df.to_csv(
    ML_TABLE_PATH,
    index=False,
)


# ============================================================
# 11. VERIFY SAVED ARTIFACT
# ============================================================

assert ML_TABLE_PATH.exists()
assert ML_TABLE_PATH.stat().st_size > 0

saved_df = pd.read_csv(
    ML_TABLE_PATH,
    nrows=5,
)

print("\nARTIFACT CHECK")
print("=" * 100)

print(f"Saved artifact: {ML_TABLE_PATH}")
print(
    f"Artifact size: "
    f"{ML_TABLE_PATH.stat().st_size / 1024 / 1024:.2f} MB"
)

display(saved_df)


# ============================================================
# 12. FINAL RESULT
# ============================================================

print("\n")
print("=" * 100)
print("NOTEBOOK 1 RESULT")
print("=" * 100)

print("SUCCESS")
print("All 9 database tables were inspected.")
print("Table row counts and keys were audited.")
print("One-to-many tables were aggregated before joining.")
print("Reviews were excluded from the ML table to avoid future leakage.")
print("The final dataset contains exactly one row per order.")
print(f"Artifact saved to: {ML_TABLE_PATH}")

In [ ]:
review_key_check = pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT review_id) AS unique_review_ids,
        COUNT(DISTINCT (review_id, order_id)) AS unique_review_order_pairs,
        COUNT(DISTINCT order_id) AS unique_orders
    FROM public.olist_order_reviews_dataset;
    """,
    engine,
)

display(review_key_check)


duplicate_reviews = pd.read_sql(
    """
    SELECT
        review_id,
        COUNT(*) AS rows_per_review_id,
        COUNT(DISTINCT order_id) AS different_orders
    FROM public.olist_order_reviews_dataset
    GROUP BY review_id
    HAVING COUNT(*) > 1
    ORDER BY rows_per_review_id DESC
    LIMIT 20;
    """,
    engine,
)

display(duplicate_reviews)

In [ ]:
review_pair_check = pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (review_id, order_id)) AS unique_pairs
    FROM public.olist_order_reviews_dataset;
    """,
    engine,
)

display(review_pair_check)